## 1. Load configuration

Paths and settings are stored in `config/dev.yaml` and `config/prod.yaml`, not in the code. Change `ENV` to `"prod"` to run on the full dataset.

In [20]:
import os, sys
sys.path.append("../src")          # lets the notebook import files from src/

from config_loader import load_config

ENV = os.environ.get("PIPELINE_ENV", "prod")   # "dev" = 2,000-row sample, "prod" = full data
config = load_config(f"../config/{ENV}.yaml")

print("Environment:", ENV)
print(config)

Environment: prod
{'environment': 'prod', 'spark': {'driver_memory': '4g', 'cores': 2, 'shuffle_partitions': 400}, 'source': {'dataset_name': 'ems_incident_dispatch', 'dataset_id': '76xm-jjuj', 'input_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data_sample/EMS_Incident_Dispatch_Data.csv'}, 'bronze': {'root_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data/bronze'}, 'silver': {'root_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data/silver'}, 'gold': {'root_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data/gold'}, 'collisions': {'dataset_name': 'motor_vehicle_collisions', 'dataset_id': 'h9gi-nx95', 'input_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data_sample/collisions_data.csv'}}


## 2. Start Spark

Create the Spark session, which is the entry point for all Spark operations. Memory, cores and shuffle partitions come from the config. `getOrCreate()` reuses a running session, so **restart the kernel** after changing these settings.

In [21]:
from pyspark.sql import SparkSession

spark_cfg = config["spark"]

spark = (
    SparkSession.builder
    .appName("ems_pipeline")
    .master(f"local[{spark_cfg['cores']}]")
    .config("spark.driver.memory", spark_cfg["driver_memory"])
    .config("spark.sql.shuffle.partitions", spark_cfg["shuffle_partitions"])
    .getOrCreate()
)

print(spark.version)
print("Master:", spark.sparkContext.master)
print("Driver memory:", spark.sparkContext.getConf().get("spark.driver.memory"))

3.5.0
Master: local[2]
Driver memory: 4g


## Performance measurement helper

Spark is lazy: transformations like `withColumn` take no time, because the work only happens at an **action** (`count`, `collect`, `write`). We wrap the actions in `timed(...)`, and each printed time covers the whole chain of work behind that action.

In [22]:
import time
from contextlib import contextmanager

@contextmanager
def timed(label):
    start = time.time()
    yield
    print(f"[{label}] {time.time() - start:.1f} s")

## 3. Read EMS Gold table 1 (only if EMS Gold is marked ready)

In [23]:
import json, os
import pandas as pd
from pyspark.sql import functions as F
from gold_collisions_rules import aggregate_collisions_month, join_ems_collisions, unmatched_collisions

gold_ems_root = os.path.abspath(os.path.join(
    "..", config["gold"]["root_path"], config["source"]["dataset_name"]
))
with open(os.path.join(gold_ems_root, "_gold_run_log.json")) as f:
    ems_log = json.load(f)
assert ems_log["status"] == "success", "EMS Gold is not ready: run the Gold notebook first"

ems = pd.read_csv(os.path.join(gold_ems_root, "response_by_borough_month.csv"))
print("EMS table 1 rows:", len(ems), "| EMS incidents:", ems["incident_count"].sum())

EMS table 1 rows: 1424 | EMS incidents: 30389767


## 4. Collisions per borough and month (from Silver)

In [24]:
silver_col_path = os.path.abspath(os.path.join(
    "..", config["silver"]["root_path"], config["collisions"]["dataset_name"], "data"
))
col_df = spark.read.parquet(silver_col_path)

with timed("Collisions by borough and month"):
    col_rows = col_df.count()
    collisions = aggregate_collisions_month(col_df)

assert collisions["collision_count"].sum() == col_rows, "Monthly collisions do not add up to Silver!"
print("Collisions in Silver:", col_rows, "| borough-month rows:", len(collisions))
print("Collision data covers years:", collisions["incident_year"].min(), "to", collisions["incident_year"].max())

[Collisions by borough and month] 0.7 s
Collisions in Silver: 2301383 | borough-month rows: 1032
Collision data covers years: 2012 to 2026


## 5. Join and reconcile (Gold table 4)

In [26]:
gold4 = join_ems_collisions(ems, collisions)
unmatched = unmatched_collisions(ems, collisions)

assert len(gold4) == len(ems), "The join changed the number of EMS rows!"
assert gold4["incident_count"].sum() == int(ems_log["silver_rows"]), "EMS incident total changed!"
matched = int(gold4["collision_count"].sum())
unmatched_total = int(unmatched["collision_count"].sum())
assert matched + unmatched_total == col_rows, "Collisions are lost or double counted!"

print("Collisions matched to an EMS borough-month:", matched)
print("Collisions with no EMS borough-month:", unmatched_total)
print(unmatched.groupby("borough_std")["collision_count"].sum().sort_values(ascending=False).head(8))
gold4[gold4["has_collision_data"]].head(8)

Collisions matched to an EMS borough-month: 1935117
Collisions with no EMS borough-month: 366266
borough_std
UNKNOWN          360286
BROOKLYN           1950
QUEENS             1680
MANHATTAN          1163
BRONX               946
STATEN ISLAND       241
Name: collision_count, dtype: int64


,borough_std,incident_year,incident_month,incident_count,dispatch_valid_count,dispatch_avg_seconds,dispatch_median_seconds,dispatch_p90_seconds,incident_response_valid_count,incident_response_avg_seconds,...,travel_median_seconds,travel_p90_seconds,dispatch_excluded_count,incident_response_excluded_count,travel_excluded_count,collision_count,persons_injured,persons_killed,has_collision_data,collisions_per_1000_ems_incidents
90,BRONX,2012,7,26302,26204,103.1,31.0,131.0,25611,570.4,...,411.0,790.0,98,691,691,1641.0,548.0,4.0,True,62.4
91,BRONX,2012,8,25568,25487,88.4,29.0,111.0,24894,549.8,...,403.0,768.0,81,674,674,1665.0,466.0,6.0,True,65.1
92,BRONX,2012,9,24117,24051,74.3,28.0,96.0,23558,526.1,...,403.0,749.0,66,559,559,1548.0,464.0,2.0,True,64.2
93,BRONX,2012,10,24791,24699,87.1,28.0,101.0,24241,542.4,...,398.0,759.0,92,550,550,1605.0,462.0,3.0,True,64.7
94,BRONX,2012,11,23153,23097,67.8,27.0,86.0,22648,515.7,...,395.0,739.0,56,505,505,1444.0,416.0,0.0,True,62.4
95,BRONX,2012,12,25126,24987,115.8,29.0,133.0,24502,590.0,...,411.0,793.0,139,624,624,1608.0,522.0,2.0,True,64.0
96,BRONX,2013,1,26492,26150,205.0,31.0,370.0,25614,707.6,...,436.0,839.0,342,878,878,1486.0,436.0,4.0,True,56.1
97,BRONX,2013,2,22399,22265,119.2,28.0,152.0,21776,603.4,...,422.0,818.0,134,623,623,1542.0,468.0,1.0,True,68.8


## 6. Quick analysis: are busy collision months also slower EMS months? (correlation is not causation; seasonality and 2020 distort monthly trends)

In [27]:
overlap = gold4[gold4["has_collision_data"] & (gold4["borough_std"] != "UNKNOWN")]
rows = []
for borough, g in overlap.groupby("borough_std"):
    rows.append({
        "borough": borough,
        "months": len(g),
        "corr_collisions_vs_median_response": round(g["collision_count"].corr(g["incident_response_median_seconds"]), 2),
    })
pd.DataFrame(rows)

,borough,months,corr_collisions_vs_median_response
0,BRONX,171,-0.37
1,BROOKLYN,171,-0.42
2,MANHATTAN,171,-0.47
3,QUEENS,171,-0.41
4,STATEN ISLAND,171,-0.56


## 7. Publish (only after the checks above passed)

In [28]:
from datetime import datetime, timezone

out_root = os.path.abspath(os.path.join("..", config["gold"]["root_path"], "ems_collisions"))
os.makedirs(out_root, exist_ok=True)
log_path = os.path.join(out_root, "_gold_run_log.json")
if os.path.exists(log_path):
    os.remove(log_path)

gold4.to_csv(os.path.join(out_root, "response_vs_collisions_borough_month.csv"), index=False)

run_log = {
    "status": "success",
    "environment": config["environment"],
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "ems_silver_rows": int(ems_log["silver_rows"]),
    "collision_silver_rows": int(col_rows),
    "rows": int(len(gold4)),
    "collisions_matched": matched,
    "collisions_unmatched": unmatched_total,
    "checks_passed": ["EMS rows and totals unchanged by the join", "matched + unmatched = collisions in Silver"],
}
with open(log_path, "w") as f:
    json.dump(run_log, f, indent=2)
print(json.dumps(run_log, indent=2))

{
  "status": "success",
  "environment": "prod",
  "completed_at_utc": "2026-11-14T13:33:17.620294+00:00",
  "ems_silver_rows": 30389767,
  "collision_silver_rows": 2301383,
  "rows": 1424,
  "collisions_matched": 1935117,
  "collisions_unmatched": 366266,
  "checks_passed": [
    "EMS rows and totals unchanged by the join",
    "matched + unmatched = collisions in Silver"
  ]
}
